# TF-IDF (Term Frequency-Inverse Document Frequency)

## Overview
TF-IDF is a numerical statistic that reflects how important a word is to a document in a collection. It addresses a key limitation of Bag of Words: not all words are equally informative.

## The Problem with Bag of Words
- Words like "the", "is", "and" appear frequently but carry little meaning
- Rare, document-specific words are more informative
- Simple word counts don't capture importance

## What is TF-IDF?

### Two Components:

#### 1. Term Frequency (TF)
**How often a term appears in a document**

Formula: $TF(t, d) = \\frac{\\text{Number of times term t appears in document d}}{\\text{Total number of terms in document d}}$

- More frequent → Higher TF
- Normalized by document length

#### 2. Inverse Document Frequency (IDF)
**How rare a term is across all documents**

Formula: $IDF(t) = \\log \\frac{\\text{Total number of documents}}{\\text{Number of documents containing term t}}$

- Appears in few documents → High IDF (rare, important)
- Appears in many documents → Low IDF (common, less informative)

### TF-IDF Score:
$TF\\text{-}IDF(t, d) = TF(t, d) \\times IDF(t)$

## Intuition:

| Word | Frequency in Doc | Appears in Docs | TF | IDF | TF-IDF | Importance |
|------|------------------|-----------------|----|----|--------|------------|
| "the" | High | Most docs | High | Low | **Low** | Not important |
| "python" | Medium | Few docs | Medium | High | **High** | Very important |
| "algorithm" | Low | Few docs | Low | High | Medium | Important |

### Key Insight:
- **High TF-IDF:** Word is frequent in this document BUT rare across other documents → Important for characterizing this document
- **Low TF-IDF:** Either infrequent in document OR common across many documents → Not distinctive

## Advantages over Bag of Words:
1. ✅ **Downweights common words** (stop words get low scores automatically)
2. ✅ **Highlights distinctive terms** that characterize documents
3. ✅ **Better for information retrieval** and document similarity
4. ✅ **No need to manually remove stop words** (they get low scores)

## Use Cases:
- **Search Engines:** Rank documents by relevance to query
- **Document Similarity:** Find similar documents
- **Keyword Extraction:** Identify important terms in documents
- **Text Classification:** Better features than raw counts
- **Recommender Systems:** Content-based filtering

In [1]:
import pandas as pd 
from sklearn.feature_extraction.text import TfidfVectorizer

## Setup
Import TfidfVectorizer from scikit-learn.

In [2]:
data = [' Most shark attacks occur about 10 feet from the beach since that is where the people are',
        'the efficiency with which he paired the socks in the drawer was quite admirable',
        'carol drank the blood as if she were a vampire',
        'giving directions that the mountains are to the west only works when you can see them',
        'the sign said there was road work ahead so he decided to speed up',
        'the gruff old man sat in the back of the bait shop grumbling to himself as he scooped out a handful of worms']

## Sample Data
Same six documents from the Bag of Words example.

In [3]:
tfidfvec = TfidfVectorizer()

## Initialize TfidfVectorizer
Create a TF-IDF vectorizer object.

In [4]:
tfidfvec_fit = tfidfvec.fit_transform(data)

## Fit and Transform
Build vocabulary and calculate TF-IDF scores for all documents.

In [5]:
tfidf_bag = pd.DataFrame(tfidfvec_fit.toarray(), columns=tfidfvec.get_feature_names_out())

## Convert to DataFrame
Create a pandas DataFrame for easy visualization.

In [6]:
print(tfidf_bag)

         10     about  admirable     ahead       are        as   attacks  \
0  0.257061  0.257061   0.000000  0.000000  0.210794  0.000000  0.257061   
1  0.000000  0.000000   0.293641  0.000000  0.000000  0.000000  0.000000   
2  0.000000  0.000000   0.000000  0.000000  0.000000  0.292313  0.000000   
3  0.000000  0.000000   0.000000  0.000000  0.222257  0.000000  0.000000   
4  0.000000  0.000000   0.000000  0.290766  0.000000  0.000000  0.000000   
5  0.000000  0.000000   0.000000  0.000000  0.000000  0.178615  0.000000   

      back     bait     beach  ...      were     west     when     where  \
0  0.00000  0.00000  0.257061  ...  0.000000  0.00000  0.00000  0.257061   
1  0.00000  0.00000  0.000000  ...  0.000000  0.00000  0.00000  0.000000   
2  0.00000  0.00000  0.000000  ...  0.356474  0.00000  0.00000  0.000000   
3  0.00000  0.00000  0.000000  ...  0.000000  0.27104  0.27104  0.000000   
4  0.00000  0.00000  0.000000  ...  0.000000  0.00000  0.00000  0.000000   
5  0.21782 

### Interpreting TF-IDF Scores:

**Compare to Bag of Words:**
- BoW: All occurrences weighted equally (just counts)
- TF-IDF: Scores reflect term importance (values between 0 and 1)

**Key Observations:**

1. **Common words** (like "the", "was") have **lower scores** across all documents
   - They appear in many documents → Low IDF → Low TF-IDF

2. **Distinctive words** have **higher scores**
   - Document 0: "shark", "attacks", "beach" have high scores (unique to this document)
   - Document 2: "vampire", "blood" have high scores (unique to this document)
   - Document 1: "efficiency", "paired", "socks" have high scores

3. **Zero values** mean the word doesn't appear in that document

4. **Magnitude differences:**
   - A word appearing 3 times in a document doesn't automatically get 3× the score
   - Rarity (IDF) matters as much as frequency (TF)

### Example Analysis:
Look at Document 0 (shark attacks):
- **High TF-IDF:** "shark", "attacks", "beach", "feet" - these characterize this document
- **Low TF-IDF:** "the", "is" - common words, less informative
- **Zero:** "vampire", "efficiency" - don't appear at all

### Practical Use:
If you want to know "What is document 0 about?", look at the highest TF-IDF scores:
- Document 0: shark, attacks, beach → Clearly about shark attacks
- Document 2: vampire, blood → About vampires

## TfidfVectorizer Parameters:
```python
TfidfVectorizer(
    max_features=100,       # Keep top 100 terms by TF-IDF
    min_df=2,               # Ignore terms in < 2 documents
    max_df=0.8,             # Ignore terms in > 80% of documents  
    ngram_range=(1,2),      # Include unigrams and bigrams
    sublinear_tf=True,      # Use log scaling for TF
    smooth_idf=True         # Add 1 to IDF to avoid zero-division
)
```

## When to Use TF-IDF vs Bag of Words:

**Use TF-IDF when:**
- ✅ Document characterization is important
- ✅ Want to find distinctive/keyword terms
- ✅ Building search/information retrieval systems
- ✅ Comparing document similarity

**Use Bag of Words when:**
- ✅ Simple classification tasks
- ✅ Already removing stop words
- ✅ Interpretability is critical (counts are intuitive)
- ✅ Working with short texts (tweets, reviews)

## Summary:
- **TF (Term Frequency):** How important is this word in THIS document?
- **IDF (Inverse Document Frequency):** How rare/unique is this word across ALL documents?
- **TF-IDF:** Balances both → Highlights words that are important to specific documents

This makes TF-IDF excellent for understanding what makes each document unique!

## View TF-IDF Matrix